[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UoA-eResearch/embedding-llms-qualitative-data-workshop/blob/main/notebooks/04-thematic-analysis.ipynb)

# Notebook 04 — Thematic Analysis and Validation

**Duration:** ~35 minutes

In notebook 03 you extracted structured facts (cross-references) from legislation. That was **deductive** — you defined what to look for in advance.

Now we go **inductive**. Instead of telling the model what to find, we ask it to discover themes on its own. This is closer to how many qualitative researchers work: you read the data, and the patterns emerge.

You will:
1. Ask the LLM to generate themes from the Privacy Act 2020
2. Run the same analysis from a different researcher's perspective and compare
3. Validate the results using a method from the published study — Jaccard similarity against parliamentary committee domains

By the end, you will understand that the prompt itself is a methodological choice — and you will have a concrete way to measure its effect.

## Setup

Run this cell first. Paste your Groq API key between the quotes.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- paste your key here
client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"

print("Setup complete.")

## Load the dataset

We use the same Privacy Act 2020 from notebook 03 — but this time we are looking for themes, not cross-references.

Run the cell below to fetch and parse the Act.

In [ ]:
import re

# Fetch the Privacy Act 2020
url = "https://legislation.govt.nz/act/public/2020/31/en/latest.xml"
response = requests.get(url)
root = etree.fromstring(response.content)

# Parse all sections
sections = []
for prov in root.findall(".//prov"):
    label_el = prov.find(".//label")
    heading_el = prov.find(".//heading")
    body_el = prov.find(".//prov.body")

    label = label_el.text.strip() if label_el is not None and label_el.text else "?"
    heading = heading_el.text.strip() if heading_el is not None and heading_el.text else ""
    text = "".join(body_el.itertext()).strip() if body_el is not None else ""

    sections.append({"label": label, "heading": heading, "text": text})

print(f"Loaded {len(sections)} sections from the Privacy Act 2020.")

For theme generation, we work with a representative sample rather than the full Act. This keeps the text within the model's context window and is standard practice — the paper we are following did the same, using LLM-generated summaries rather than raw text.

The cell below builds a corpus from the first 20 substantive sections. Each section is truncated to 400 characters to keep the total manageable.

In [ ]:
# Build a corpus from sections 3–22 (the first 20 substantive sections)
sample = sections[2:22]

corpus = ""
for s in sample:
    snippet = s["text"][:400]
    corpus += f"Section {s['label']} ({s['heading']}): {snippet}\n\n"

print(f"Corpus: {len(sample)} sections, {len(corpus)} characters")
print()
print("Sections included:")
for s in sample:
    print(f"  s{s['label']}: {s['heading']}")

## Where this fits in the research workflow

> **Research context:** The published study on NZ legislative networks (Ardekani et al., 2026) proposed a method they called "LLM-assisted LDA". Instead of running topic models on raw legislative text — full of procedural boilerplate like "notwithstanding section" and "subsection (1)(a)" — they first used an LLM to generate compact summaries and keywords for each Act. This **semantic abstraction** produced topics with much higher coherence and cleaner separation.
>
> What you are about to do is a tutorial-level version of the same idea: ask an LLM to identify themes from legislation, then validate those themes against an external reference.

This corresponds to **Steps 4, 5, and 6** of the workflow:

| Step | What it does | What you will do |
|------|-------------|------------------|
| 4. Semantic enrichment | Use an LLM to summarise and extract meaning | Exercise a — generate themes and keywords |
| 5. Analysis | Find patterns and themes | Exercise b — compare themes from different perspectives |
| 6. Interpretation | Check findings against reality | Exercise c — Jaccard validation against committees |

## Exercise a — Inductive theme generation

**Inductive coding** means you do not define categories in advance. You let the data speak. The LLM reads the text and proposes themes based on what it finds.

We use a **neutral prompt** — no particular perspective, no leading framing. We ask the model to focus on substantive themes (what the Act is about) and exclude procedural language (section numbers, subsection references).

Run the cell below.

In [ ]:
neutral_response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a qualitative research assistant. You identify themes in legislative text."
        },
        {
            "role": "user",
            "content": f"""Identify 5 to 7 recurring themes in the following legislative sections.
Focus on substantive themes and obligations. Exclude procedural boilerplate
(such as section numbering, commencement dates, and administrative language).

For each theme, provide:
- A short theme name
- A one-sentence description
- 10 to 15 keywords associated with that theme

Return your answer as a JSON array, e.g.:
[
  {{
    "theme": "Theme Name",
    "description": "One sentence description.",
    "keywords": ["word1", "word2", "word3"]
  }}
]

Legislative text:
{corpus}"""
        }
    ]
)

neutral_raw = neutral_response.choices[0].message.content.strip()

# Parse the JSON
try:
    neutral_themes = json.loads(neutral_raw)
except json.JSONDecodeError:
    # Sometimes the model wraps JSON in markdown code fences — strip them
    cleaned = neutral_raw.strip("`").strip()
    if cleaned.startswith("json"):
        cleaned = cleaned[4:].strip()
    neutral_themes = json.loads(cleaned)

print(f"The model identified {len(neutral_themes)} themes:\n")
for i, t in enumerate(neutral_themes, 1):
    print(f"{i}. {t['theme']}")
    print(f"   {t['description']}")
    print(f"   Keywords: {', '.join(t['keywords'])}")
    print()

Look at the themes. Do they make sense for a privacy law? Are any too broad or too narrow? We will come back to this question.

For now, let's collect all the keywords into a single set — we will use them for comparison later.

In [ ]:
# Collect all keywords from the neutral themes
neutral_keywords = set()
for t in neutral_themes:
    for kw in t["keywords"]:
        neutral_keywords.add(kw.lower())

print(f"Total unique keywords from neutral analysis: {len(neutral_keywords)}")
print(sorted(neutral_keywords))

## Exercise b — Persona experiment

In notebook 02 you saw that the `system` message (the role) shapes the model's output. Now we test this deliberately.

We run the **exact same prompt** on the **exact same data**, but with a positioned persona instead of a neutral one.

The question is: does the researcher's perspective change what themes the model finds?

In [ ]:
# Choose a positioned persona.
# You can change this to any of the options below, or write your own.
#
# Options:
#   "You are a privacy rights advocate analysing surveillance risks in this legislation."
#   "You are a business compliance officer assessing regulatory burden."
#   "You are a Māori data sovereignty researcher examining indigenous data rights."
#   "You are a journalist investigating government transparency."

positioned_persona = "You are a privacy rights advocate analysing surveillance risks in this legislation."

positioned_response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": positioned_persona
        },
        {
            "role": "user",
            "content": f"""Identify 5 to 7 recurring themes in the following legislative sections.
Focus on substantive themes and obligations. Exclude procedural boilerplate
(such as section numbering, commencement dates, and administrative language).

For each theme, provide:
- A short theme name
- A one-sentence description
- 10 to 15 keywords associated with that theme

Return your answer as a JSON array, e.g.:
[
  {{
    "theme": "Theme Name",
    "description": "One sentence description.",
    "keywords": ["word1", "word2", "word3"]
  }}
]

Legislative text:
{corpus}"""
        }
    ]
)

positioned_raw = positioned_response.choices[0].message.content.strip()

try:
    positioned_themes = json.loads(positioned_raw)
except json.JSONDecodeError:
    cleaned = positioned_raw.strip("`").strip()
    if cleaned.startswith("json"):
        cleaned = cleaned[4:].strip()
    positioned_themes = json.loads(cleaned)

print(f"Positioned persona: {positioned_persona}")
print(f"The model identified {len(positioned_themes)} themes:\n")
for i, t in enumerate(positioned_themes, 1):
    print(f"{i}. {t['theme']}")
    print(f"   {t['description']}")
    print(f"   Keywords: {', '.join(t['keywords'])}")
    print()

In [ ]:
# Collect all keywords from the positioned analysis
positioned_keywords = set()
for t in positioned_themes:
    for kw in t["keywords"]:
        positioned_keywords.add(kw.lower())

print(f"Total unique keywords from positioned analysis: {len(positioned_keywords)}")
print(sorted(positioned_keywords))

### Compare the two analyses

Let's see where the keyword sets overlap and where they diverge.

In [ ]:
# Compare keyword sets
shared = neutral_keywords & positioned_keywords
neutral_only = neutral_keywords - positioned_keywords
positioned_only = positioned_keywords - neutral_keywords

print(f"Keywords shared by both: {len(shared)}")
print(f"Keywords only in neutral: {len(neutral_only)}")
print(f"Keywords only in positioned: {len(positioned_only)}")
print()

if shared:
    print("Shared keywords:")
    print(f"  {', '.join(sorted(shared))}")
    print()

if neutral_only:
    print("Neutral only:")
    print(f"  {', '.join(sorted(neutral_only))}")
    print()

if positioned_only:
    print("Positioned only:")
    print(f"  {', '.join(sorted(positioned_only))}")

Look at the "positioned only" keywords. Are they about surveillance, data collection, and rights — the topics the persona was primed to notice? That is not a bug. It is the whole point.

**Discussion questions:**
- Which prompt is "neutral"? Is either truly neutral?
- If the outputs differ, which is right?
- What does your methods section need to say about the prompt you used?

The answer: differences are not right or wrong. They reflect different research questions. A privacy rights advocate and a business compliance officer will notice different things in the same text — and so will an LLM playing those roles. Naming that a positioned prompt finds its persona's keywords is the methodological insight, not a flaw to hide.

## Exercise c — Jaccard validation

You have two sets of keywords. But how do you know if they are meaningful? One way is to compare them against an **external reference** — something that was not generated by the model.

The paper did this by comparing their computationally derived topic clusters against real-world **parliamentary committee domains**. New Zealand's Parliament organises legislation by subject area — Justice, Health, Environment, and so on. Each committee has a recognisable set of terms.

We use **Jaccard similarity** to measure how much two sets overlap. The formula is simple:

$$\text{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- 0 means no overlap at all
- 1 means the sets are identical

In plain English: count the words both sets share, divide by the total number of unique words across both sets.

In [ ]:
# Jaccard similarity function
def jaccard_similarity(set_a, set_b):
    """Calculate Jaccard similarity between two sets of strings."""
    a = set(w.lower() for w in set_a)
    b = set(w.lower() for w in set_b)
    intersection = a & b
    union = a | b
    if not union:
        return 0.0
    return len(intersection) / len(union)

print("Jaccard similarity function defined.")
print()

# Quick test: two identical sets should score 1.0
print(f"Test (identical sets): {jaccard_similarity({'a','b','c'}, {'a','b','c'}):.2f}")
print(f"Test (no overlap):    {jaccard_similarity({'a','b'}, {'c','d'}):.2f}")
print(f"Test (partial):       {jaccard_similarity({'a','b','c'}, {'b','c','d'}):.2f}")

### Committee keyword sets

Below is a dictionary of keywords associated with six New Zealand parliamentary select committees. These are drawn from the committees' published terms of reference and the types of legislation they review.

In [ ]:
# Parliamentary select committee keyword sets
COMMITTEE_KEYWORDS = {
    "Justice Committee": [
        "criminal", "justice", "court", "offence", "penalty",
        "prosecution", "enforcement", "police", "imprisonment",
        "tribunal", "legal", "rights", "appeal", "sentence", "conviction"
    ],
    "Health Committee": [
        "health", "medical", "treatment", "patient", "clinical",
        "disease", "mental", "disability", "care", "hospital",
        "practitioner", "pharmaceutical", "wellbeing", "public health",
        "safety"
    ],
    "Environment Committee": [
        "environment", "resource", "land", "water", "conservation",
        "climate", "emissions", "biodiversity", "sustainable",
        "pollution", "ecological", "planning", "consent", "impact",
        "natural"
    ],
    "Finance and Expenditure Committee": [
        "financial", "revenue", "tax", "expenditure", "budget",
        "fiscal", "economic", "payment", "fund", "investment",
        "commercial", "income", "cost", "penalty", "levy"
    ],
    "Governance and Administration Committee": [
        "privacy", "information", "data", "personal", "agency",
        "public", "government", "official", "disclosure", "access",
        "transparency", "accountability", "complaint", "commissioner",
        "record"
    ],
    "Social Services and Community Committee": [
        "social", "community", "welfare", "family", "housing",
        "poverty", "employment", "education", "youth", "elderly",
        "disability", "support", "benefit", "vulnerable", "protection"
    ]
}

print(f"Defined {len(COMMITTEE_KEYWORDS)} committee keyword sets.")
for name, kws in COMMITTEE_KEYWORDS.items():
    print(f"  {name}: {len(kws)} keywords")

### Compare theme keywords against committees

Now we measure how much the keywords from each analysis (neutral and positioned) overlap with each committee's keyword set.

The Privacy Act should align most strongly with the **Governance and Administration Committee** — that is the committee that actually reviews privacy legislation.

In [ ]:
# Compare neutral keywords against each committee
print("NEUTRAL analysis — Jaccard similarity with each committee:")
print("=" * 60)
neutral_scores = {}
for committee, kws in COMMITTEE_KEYWORDS.items():
    score = jaccard_similarity(neutral_keywords, kws)
    neutral_scores[committee] = score
    bar = "█" * int(score * 40)
    print(f"  {committee:45s} {score:.3f} {bar}")

best_neutral = max(neutral_scores, key=neutral_scores.get)
print(f"\nBest match: {best_neutral} ({neutral_scores[best_neutral]:.3f})")

In [ ]:
# Compare positioned keywords against each committee
print(f"POSITIONED analysis — Jaccard similarity with each committee:")
print(f"Persona: {positioned_persona}")
print("=" * 60)
positioned_scores = {}
for committee, kws in COMMITTEE_KEYWORDS.items():
    score = jaccard_similarity(positioned_keywords, kws)
    positioned_scores[committee] = score
    bar = "█" * int(score * 40)
    print(f"  {committee:45s} {score:.3f} {bar}")

best_positioned = max(positioned_scores, key=positioned_scores.get)
print(f"\nBest match: {best_positioned} ({positioned_scores[best_positioned]:.3f})")

### Side-by-side comparison

Let's put both sets of scores in one table so you can see how the persona shifted the results.

In [ ]:
# Side-by-side comparison
print(f"{'Committee':45s} {'Neutral':>8s} {'Positioned':>10s} {'Shift':>8s}")
print("=" * 75)
for committee in COMMITTEE_KEYWORDS:
    n = neutral_scores[committee]
    p = positioned_scores[committee]
    shift = p - n
    arrow = "↑" if shift > 0.005 else "↓" if shift < -0.005 else "–"
    print(f"  {committee:43s} {n:8.3f} {p:10.3f} {shift:+8.3f} {arrow}")

**What to look for:**

- Did the positioned persona move the Jaccard score **toward its expected committee**? A privacy rights advocate should push scores higher for Governance and Administration (and possibly Justice).
- Did it pull scores **away** from unrelated committees like Environment or Health?
- How large is the shift? Even a small, consistent shift matters — it means the prompt framing changed which themes the model found in the same data.

> "The paper validated their clusters this way — comparing computationally derived communities against parliamentary committee domains. You just did the same."

**Discussion questions:**
- Did your positioned prompt move the needle toward its target committee? What does that mean for validity?
- If you reported only the positioned results, would a reader know the prompt shaped the output?
- What would you write in a methods section to be transparent about this?

## Optional: Adversarial prompting

If you have time, try this stretch exercise. **Adversarial prompting** means asking the model to argue against its own output. It is a way to stress-test themes — the ones that survive scrutiny are more robust.

Run the cell below to send the neutral themes back to the model with a challenge.

In [ ]:
# Format the neutral themes as a string for the adversarial prompt
theme_summary = ""
for t in neutral_themes:
    theme_summary += f"- {t['theme']}: {t['description']}\n"

adversarial_response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a critical reviewer of qualitative research methods."
        },
        {
            "role": "user",
            "content": f"""A researcher identified the following themes in New Zealand's Privacy Act 2020:

{theme_summary}
Argue against these themes. For each one, explain:
1. Is the theme too broad, too narrow, or overlapping with another?
2. Is there evidence in privacy legislation that contradicts this theme?
3. What theme might be missing entirely?

Be specific and critical."""
        }
    ]
)

print("Adversarial critique:")
print("=" * 60)
print(adversarial_response.choices[0].message.content)

Read the critique. Which themes survived? Which ones did the model argue were too broad or overlapping?

This does not prove the themes are right or wrong. But it gives you a structured way to question your own analysis — which is what good qualitative research does.

## What you accomplished

In this notebook you:

- Generated themes **inductively** from real legislative data using an LLM (**Step 4: Semantic Enrichment**)
- Ran the same analysis with a **positioned persona** and measured how the prompt changed the results (**Step 5: Analysis**)
- Validated theme keywords against parliamentary committee domains using **Jaccard similarity** (**Step 6: Interpretation**)
- Optionally, stress-tested themes using **adversarial prompting**

The key insight: the prompt is not a neutral container. It is a methodological choice. The role you assign the model, the instructions you give it, and the framing you use all shape what themes emerge — just as a human researcher's background shapes what they notice. The difference is that with an LLM, you can measure the effect precisely.

You now know four named validation techniques:
1. **Consistency check** (notebook 02 — temperature)
2. **Cross-method comparison** (notebook 03 — regex vs LLM)
3. **Jaccard validation** (this notebook — themes vs committees)
4. **Adversarial prompting** (this notebook — challenging the model's own output)

**Next up:** After the break, notebook 05 applies the same pipeline to images — proving that these techniques work across data types, not just text.